# Agent walkthrough — observation, weather, assimilation

Every decision the three agents can make, one at a time, with the state shown before and
after. Nothing here touches FARSITE, the network or `data/`: perimeters are circles written
to a temporary directory, weather is a literal dict, and the forecast ensemble is random
numbers scaled to a chosen innovation.

The companion suite `test_agents.py` asserts the same behaviour (61 tests, ~1 s). This
notebook is the readable version — run it top to bottom to see what each agent does and
why.

| Section | Agent | What it shows |
|---|---|---|
| 1 | `observation` | accept / reject, every rejection rule, the first-observation guardrail |
| 2 | `weather` | window selection, the three conditions, cache rotation, fetch-failure guardrail |
| 3 | `assimilation` | all four EnKF actions, priority order, the no-state guardrail |
| 4 | — | how `Signals` fills up across the three agents |
| 5 | all three | a 4-cycle replay: Fixed (baseline) vs Rule (agentic), identical inputs |

**Prerequisite:** `bash setup_env.sh` from the repo root, then select the
**Python (wildfire-twin)** kernel.

## Setup

In [1]:
from __future__ import annotations

import tempfile
from datetime import datetime, timedelta, timezone
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import pyproj
from shapely.geometry import Point

from orchestrator import checks
from orchestrator.agents.assimilation import (
    DRIFT_HISTORY_LEN, make_assimilation_agent,
    FixedPolicy as AsFixed, RulePolicy as AsRule,
)
from orchestrator.agents.observation import (
    compute_signals, make_observation_agent,
    FixedPolicy as ObsFixed, RulePolicy as ObsRule,
)
from orchestrator.agents.weather import (
    FALLBACK_DIRECTIONS, FALLBACK_SPEEDS, forecast_window, make_weather_agent,
    FixedPolicy as WxFixed, RulePolicy as WxRule,
)
from orchestrator.graph import build_pipeline, run_pipeline
from orchestrator.placeholders import placeholder_nodes
from orchestrator.state import (
    BASELINE, VSIZE_PRESETS, ForecastPlan, ObservationEvent, OperatorRequest,
    Signals, WeatherVerdict, initial_state,
)

pd.set_option("display.max_colwidth", 70)
print("imports ok")

imports ok


### Perimeters

The agents read perimeters off disk (`event.path`), so the helper below writes circles to a
temp directory. A circle is enough: every check is about **relative** area and containment.
They are centred on the real Border 2 ignition point so the coordinates are in the right
part of EPSG:5070.

In [2]:
TMP = Path(tempfile.mkdtemp(prefix="agent_demo_"))

# Border 2 ignition point (lat/lon) -> EPSG:5070 metres
_to5070 = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:5070", always_xy=True)
CX, CY = _to5070.transform(-116.85708741551528, 32.59588662537945)


def perimeter(name: str, radius_m: float, dx: float = 0.0, dy: float = 0.0) -> str:
    '''Write a circular perimeter as GeoJSON in EPSG:5070 and return its path.'''
    path = TMP / f"{name}.geojson"
    geom = Point(CX + dx, CY + dy).buffer(radius_m)
    gpd.GeoDataFrame(geometry=[geom], crs="EPSG:5070").to_file(path, driver="GeoJSON")
    return str(path)


def area_ha(path: str) -> float:
    return gpd.read_file(path).to_crs(epsg=5070).geometry.iloc[0].area / 10_000


# A small family of perimeters to draw on through the notebook
P_PREV     = perimeter("prev",     2000)                 # the established perimeter
P_GROWN    = perimeter("grown",    2600)                 # plausible 3 h growth
P_SHRUNK   = perimeter("shrunk",    900)                 # fires do not shrink
P_SHIFTED  = perimeter("shifted",  2000, dx=4000)        # right size, wrong place
P_ENORMOUS = perimeter("enormous", 60000)                # gross error

for n, p in [("prev", P_PREV), ("grown", P_GROWN), ("shrunk", P_SHRUNK),
             ("shifted", P_SHIFTED), ("enormous", P_ENORMOUS)]:
    print(f"  {n:9s} {area_ha(p):10,.0f} ha")

  prev           1,255 ha
  grown          2,120 ha
  shrunk           254 ha
  shifted        1,255 ha
  enormous   1,129,157 ha


### Weather, display helpers and a base state

In [3]:
CALM     = {"wind_speeds": [8.0, 9.0],   "wind_directions": [220.0, 225.0], "used_fallback": False}
SHIFT    = {"wind_speeds": [30.0, 32.0], "wind_directions": [40.0, 45.0],   "used_fallback": False}
HIGH     = {"wind_speeds": [30.0, 31.0], "wind_directions": [220.0, 225.0], "used_fallback": False}
FALLBACK = {"wind_speeds": list(FALLBACK_SPEEDS),
            "wind_directions": list(FALLBACK_DIRECTIONS), "used_fallback": True}

T0 = datetime(2025, 1, 23, 12, 0, tzinfo=timezone.utc)   # Border 2, day two


def event(i: int = 1, path: str = P_GROWN, **kw) -> ObservationEvent:
    '''An observation arriving 5 min after it was made, 3 h after event i-1.'''
    kw.setdefault("obs_time", T0 + timedelta(hours=3 * i))
    kw.setdefault("arrival_time", kw["obs_time"] + timedelta(minutes=5))
    kw.setdefault("source", "mapped_perimeter")
    kw.setdefault("feature_id", "0")
    return ObservationEvent(path=path, **kw)


def base_state(current: ObservationEvent | None = None, **over) -> dict:
    '''Mid-replay state: one observation in hand, a previous perimeter on record.'''
    ev = current or event(1)
    st = dict(initial_state([ev], OperatorRequest(deadline_min=60), str(TMP / "state.npz")))
    st.update(current=ev, pending=[], clock=ev.arrival_time,
              prev_obs_path=P_PREV, step=1,
              cache={"good_obs": [event(0, P_PREV).model_dump(mode="json")],
                     "weather_last": CALM})
    st.update(over)
    return st


def show(verdict, label: str = "") -> None:
    fields = {k: v for k, v in verdict.model_dump().items() if k != "reason"}
    head = " ".join(f"{k}={v}" for k, v in fields.items())
    print(f"{label}{head}\n    {verdict.reason}")


def signal_row(sig: Signals) -> dict:
    return {"innovation": round(sig.normalized_innovation, 2),
            "contains_prev": sig.contains_previous,
            "growth_ok": sig.growth_plausible,
            "firms": sig.agrees_with_firms,
            "age_min": round(sig.obs_age_min),
            "wind_shift": sig.wind_shift,
            "streak": sig.consecutive_disagreements,
            "FAILED": sig.independent_check_failed}


print(f"temp dir: {TMP}")

temp dir: /tmp/agent_demo_6w2mud9c


---

## 1. Observation agent

Runs first in every cycle and answers one question: **do we trust this perimeter?**
`accept` sends it on to the forecast; `reject` routes to `synthesize`, which replaces it
with a perimeter grown from the last accepted one — a rejected observation never skips the
forecast.

It also picks the `ForecastPlan`, because ensemble size and resolution depend on the
deadline and the source, neither of which needs the forecast.

### 1.1 `FixedPolicy` — the control condition

In [4]:
st = base_state()
out = make_observation_agent(ObsFixed())(st)

show(out["obs_verdict"], "verdict  ")
print()
print(f"plan     N={out['plan'].ensemble_size} "
      f"res={out['plan'].resolution_m} m workers={out['plan'].workers}")
print(f"         {out['plan'].reason}")
print()
print("matches BASELINE:",
      (out["plan"].ensemble_size, out["plan"].resolution_m, out["plan"].workers)
      == (BASELINE["ensemble_size"], BASELINE["resolution_m"], BASELINE["workers"]))

verdict  action=accept override_guardrail=False
    FixedPolicy: accept all observations unconditionally (baseline).

plan     N=300 res=150 m workers=12
         FixedPolicy: baseline settings (N=300, 150 m, 12 workers).

matches BASELINE: True


`FixedPolicy` accepts unconditionally with baseline settings, which is what the
`outputs/Border_2_Real` run did. It is the control every agentic run is measured against.

It still computes the signals, though — so the baseline run produces the same diagnostics
without acting on them:

In [5]:
pd.DataFrame([signal_row(out["signals"])]).T.rename(columns={0: "value"})

,value
innovation,0.0
contains_prev,True
growth_ok,True
firms,None
age_min,5
wind_shift,False
streak,0
FAILED,False


`innovation` is `0.0` because nothing has been forecast yet — the observation node runs
first, so there is no ensemble to measure against. It is a placeholder, not a measurement;
the assimilation agent fills it in later (section 4). `firms` is `None` throughout this
notebook because no FIRMS CSV is wired up, and `None` means "cannot judge", which the rules
treat differently from `False`.

### 1.2 `RulePolicy` — the checks that matter

Three rejection rules, any one sufficient:

1. a forecast-independent check failed **and** no wind shift explains it
2. the observation is older than `max_age_min` (default 360)
3. it carries an injected fault **and** `reject_injected=True` (test mode; off by default)

Start with the clean case:

In [6]:
out = make_observation_agent(ObsRule())(base_state())
show(out["obs_verdict"], "grown perimeter, calm wind  ")

grown perimeter, calm wind  action=accept override_guardrail=False
    All checks passed.


#### Rejection 1a — the perimeter shrank

In [7]:
out = make_observation_agent(ObsRule())(base_state(event(1, P_SHRUNK)))
show(out["obs_verdict"], "")
print()
pd.DataFrame([signal_row(out["signals"])]).T.rename(columns={0: "value"})

action=reject override_guardrail=False
    Reject: perimeter does not contain the previous one.



,value
innovation,0.0
contains_prev,False
growth_ok,True
firms,None
age_min,5
wind_shift,False
streak,0
FAILED,True


`contains_previous` is False, so `independent_check_failed` goes True and the observation is
rejected. This is the check that catches a displaced perimeter too — right size, wrong place:

In [8]:
for label, path in [("shrunk  ", P_SHRUNK), ("shifted ", P_SHIFTED), ("enormous", P_ENORMOUS)]:
    o = make_observation_agent(ObsRule())(base_state(event(1, path)))
    s = o["signals"]
    print(f"{label} contains_prev={str(s.contains_previous):5s} "
          f"growth_ok={str(s.growth_plausible):5s} -> {o['obs_verdict'].action}")

shrunk   contains_prev=False growth_ok=True  -> reject
shifted  contains_prev=False growth_ok=True  -> reject
enormous contains_prev=True  growth_ok=False -> reject


Note which check fires for which fault. Displacement trips `contains_previous`; only the
60 km perimeter trips `growth_plausible`. That split is deliberate — `growth_plausible` is a
sanity check for gross errors (wrong fire, units mix-up, decimal slip), not a tight bound on
spread, because a false rejection throws away real data. Its ceiling scales with wind:

In [9]:
pd.DataFrame({"wind (mph)": [0, 5, 10, 15, 25, 40],
              "max growth (m/min)": [checks.spread_ceiling(w) for w in [0, 5, 10, 15, 25, 40]]}
             ).set_index("wind (mph)")

,max growth (m/min)
wind (mph),
0,30.0
5,80.0
10,130.0
15,180.0
25,280.0
40,300.0


#### Rejection 2 — the observation is stale

In [10]:
stale = base_state()
stale["clock"] = stale["current"].obs_time + timedelta(hours=10)   # 600 min old

out = make_observation_agent(ObsRule())(stale)
show(out["obs_verdict"], "")
print(f"\nobs_age_min = {out['signals'].obs_age_min:.0f}  (threshold 360)")

action=reject override_guardrail=False
    Reject: observation is 600 min old (> 360 min threshold).

obs_age_min = 600  (threshold 360)


#### The wind-shift relaxation

The one case where a failed independent check does *not* reject. A sharp wind shift can move
a fire front far enough that a legitimate perimeter looks wrong, so the independent checks
are relaxed when one is detected.

Same shrunken perimeter as before, but now the wind has shifted:

In [11]:
shifted_wind = base_state(event(1, P_SHRUNK),
                          cache={"good_obs": [event(0, P_PREV).model_dump(mode="json")],
                                 "weather_prev": CALM, "weather_last": SHIFT})

out = make_observation_agent(ObsRule())(shifted_wind)
print("wind_shift:", out["signals"].wind_shift, " | independent_check_failed:",
      out["signals"].independent_check_failed)
show(out["obs_verdict"], "\n")

wind_shift: True  | independent_check_failed: True

action=accept override_guardrail=False
    Independent checks flagged issues but wind_shift detected — accepting (wind shift can cause legitimate perimeter jumps).


`wind_shift` here is **one cycle lagged** — it compares `cache["weather_prev"]` with
`cache["weather_last"]`, because the observation node runs before the weather node, so the
current window has not been fetched yet.

That lag is not a compromise. This observation reports growth over the interval that just
*ended*, so the question worth asking is "did the wind shift during the interval that
produced this perimeter?" — which is exactly what the lagged comparison answers. The weather
agent later overwrites the field with the forward-looking value for the assimilation agent
(section 4).

#### The guardrail — the first observation is always accepted

With no previous perimeter, containment and growth are uninformative, and rejecting the
first observation would leave the state uninitialised. The node overrides the policy and
records it.

In [12]:
first = base_state(event(1, P_SHRUNK), prev_obs_path=None, cache={})
first["clock"] = first["current"].obs_time + timedelta(hours=10)   # stale AND shrunken

out = make_observation_agent(ObsRule())(first)
show(out["obs_verdict"], "")
print(f"\noverride_guardrail      {out['obs_verdict'].override_guardrail}")
print(f"logged as overridden    {out['log'][0].overridden_by_guardrail}")

action=accept override_guardrail=True
    Guardrail: no previous observation to compare against; must accept the first observation to initialise the state.

override_guardrail      True
logged as overridden    True


### 1.3 The plan scales with the deadline

`RulePolicy` trades ensemble size for wall-clock time. A tighter budget buys fewer members
and a coarser grid.

In [13]:
rows = []
for deadline in (10, 20, 30, 45, 60, 120):
    for source in ("mapped_perimeter", "firms_modis"):
        ev = event(1, P_GROWN, source=source)
        st = base_state(ev, request=OperatorRequest(deadline_min=deadline))
        plan = make_observation_agent(ObsRule())(st)["plan"]
        rows.append({"deadline_min": deadline, "source": source,
                     "N": plan.ensemble_size, "resolution_m": plan.resolution_m})
pd.DataFrame(rows).pivot(index="deadline_min", columns="source",
                         values=["N", "resolution_m"])

N                  resolution_m                 
source       firms_modis mapped_perimeter  firms_modis mapped_perimeter
deadline_min                                                           
10                    50               50          300              300
20                   150              150          300              150
30                   150              150          300              150
45                   150              150          300              150
60                   300              300          300              150
120                  300              300          300              150

---

## 2. Weather agent

Fetches the wind for the interval about to be forecast and flags conditions the rest of the
pipeline should treat differently. It is the only node that talks to the weather service, so
it owns the weather cache:

| cache key | meaning |
|---|---|
| `weather_last` | the window just fetched — `synthesize` grows perimeters downwind with it |
| `weather_prev` | what `weather_last` held on entry — the next cycle's lagged `wind_shift` |

Its `fetch` is injectable, which is how this notebook runs without a network.

### 2.1 Which interval gets forecast

In [14]:
# First cycle: nothing accepted yet, so default to a 1 h window ending at the observation
first = base_state(cache={})
s, e = forecast_window(first)
print(f"no previous accepted obs : {s:%H:%M} -> {e:%H:%M}  ({e - s})")

# Later cycle: span from the last accepted observation to this one
s, e = forecast_window(base_state())
print(f"good_obs populated       : {s:%H:%M} -> {e:%H:%M}  ({e - s})")

no previous accepted obs : 14:00 -> 15:00  (1:00:00)
good_obs populated       : 12:00 -> 15:00  (3:00:00)


### 2.2 `FixedPolicy` records a shift without acting on it

The control condition has to stay comparable, so it never changes behaviour — but it still
reports what it saw, which keeps the baseline's diagnostics usable.

In [15]:
out = make_weather_agent(WxFixed(), fetch=lambda s, e: SHIFT)(base_state())
show(out["weather_verdict"], "")
print("\naction stays 'normal', wind_shift is recorded as True")

action=normal used_fallback=False wind_shift=True
    FixedPolicy: mean wind 31.0 mph; no conditional handling (baseline).

action stays 'normal', wind_shift is recorded as True


### 2.3 `RulePolicy` — three conditions

In [16]:
rows = []
for label, prev, now in [
    ("steady wind",        CALM, CALM),
    ("direction reverses", CALM, SHIFT),
    ("no RAWS data",       CALM, FALLBACK),
]:
    st = base_state(cache={"weather_last": prev})
    v = make_weather_agent(WxRule(), fetch=lambda s, e, w=now: w)(st)["weather_verdict"]
    rows.append({"scenario": label, "action": v.action,
                 "wind_shift": v.wind_shift, "used_fallback": v.used_fallback,
                 "reason": v.reason})
pd.DataFrame(rows).set_index("scenario")

,action,wind_shift,used_fallback,reason
scenario,,,,
steady wind,normal,False,False,Conditions steady; mean wind 8.5 mph.
direction reverses,wind_shift_detected,True,False,Wind shifted sharply since the last window (> 45° or > 10 mph); me...
no RAWS data,fallback_weather,False,True,No RAWS data for this window; firemap returned its static fallback...


`wind_shift_detected` outranks `fallback_weather`: a shift changes how the *assimilation*
behaves, while fallback weather only widens the uncertainty.

### 2.4 The cache rotates so the next cycle can compare

In [17]:
st = base_state(cache={})

print("cycle 1 — nothing to compare against yet")
out1 = make_weather_agent(WxRule(), fetch=lambda s, e: CALM)(st)
print("  verdict      :", out1["weather_verdict"].action)
print("  cache keys   :", sorted(k for k in out1["cache"] if k.startswith("weather")))

print("\ncycle 2 — the wind has turned")
st2 = {**st, "cache": out1["cache"]}
out2 = make_weather_agent(WxRule(), fetch=lambda s, e: SHIFT)(st2)
print("  verdict      :", out2["weather_verdict"].action)
print("  weather_prev :", out2["cache"]["weather_prev"]["wind_directions"], "(was weather_last)")
print("  weather_last :", out2["cache"]["weather_last"]["wind_directions"])

cycle 1 — nothing to compare against yet
  verdict      : normal
  cache keys   : ['weather_last']

cycle 2 — the wind has turned
  verdict      : wind_shift_detected
  weather_prev : [220.0, 225.0] (was weather_last)
  weather_last : [40.0, 45.0]


### 2.5 Guardrail — a fetch failure never kills the replay

The weather service is the one external dependency in the loop. If it raises, the node falls
back to static values, flags the verdict as `fallback_weather` whatever the policy decided,
and the cycle continues.

In [18]:
def broken_service(start, end):
    raise ConnectionError("firemap: read timeout after 30s")

out = make_weather_agent(WxFixed(), fetch=broken_service)(base_state())
show(out["weather_verdict"], "")
print(f"\nlogged as overridden  {out['log'][0].overridden_by_guardrail}")
print(f"fetch_error recorded  {out['log'][0].inputs['fetch_error']}")
print(f"replay continues on   {out['cache']['weather_last']['wind_speeds']} mph")

action=fallback_weather used_fallback=True wind_shift=False
    Guardrail: weather fetch failed (ConnectionError: firemap: read timeout after 30s); using the static fallback. Forecast uncertainty is higher.

logged as overridden  True
fetch_error recorded  ConnectionError: firemap: read timeout after 30s
replay continues on   [10.0] mph


### 2.6 High wind can refine the plan

Above `high_wind_mph` the front moves far within one FARSITE sub-step, so the plan drops to a
finer grid — but only when the deadline has room for it.

In [19]:
rows = []
for deadline in (20, 60):
    for label, w in [("calm (8 mph)", CALM), ("high (30 mph)", HIGH)]:
        st = base_state(request=OperatorRequest(deadline_min=deadline),
                        plan=ForecastPlan(ensemble_size=300, resolution_m=150, workers=12),
                        cache={"weather_last": w})   # same wind both sides: no shift
        out = make_weather_agent(WxRule(), fetch=lambda s, e, x=w: x)(st)
        rows.append({"deadline_min": deadline, "wind": label,
                     "resolution_m": out["plan"].resolution_m if "plan" in out else 150,
                     "changed": "plan" in out})
pd.DataFrame(rows).set_index(["deadline_min", "wind"])

resolution_m  changed
deadline_min wind                                
20           calm (8 mph)            150    False
             high (30 mph)           150    False
60           calm (8 mph)            150    False
             high (30 mph)            75     True

---

## 3. Assimilation agent

The agent with real consequences: it picks how the observation moves the fire state.
Four actions, evaluated in priority order.

| Priority | Action | When |
|---|---|---|
| 1 | `skip` | an independent check rejects the observation **and** no wind shift explains it |
| 2 | `reinitialize_from_obs` | the forecast has disagreed `drift_steps` times running — the *model* is wrong |
| 3 | `inflate_and_assimilate` | the observation is surprising but believable |
| 4 | `assimilate` | everything else |

`skip` is last-resort: discarding data is only justified when something other than the
forecast says the observation is bad. `vsize_m` comes from `VSIZE_PRESETS[source]`, so a
synthetic perimeter is trusted less than a mapped one.

This is the one agent that needs the forecast ensemble, since `normalized_innovation`
measures the observation against the spread of FARSITE members. The loader is injectable,
so below it is random numbers scaled to a chosen innovation.

In [20]:
def ensemble_with_innovation(target: float, source: str = "mapped_perimeter", seed: int = 0):
    '''A (forecast, obs) pair whose normalized_innovation is about `target`.'''
    rng = np.random.default_rng(seed)
    n_states, n_members = 40, 50
    Z = rng.normal(0, 100, (n_states, n_members))
    spread = np.linalg.norm(Z - Z.mean(axis=1, keepdims=True), axis=0).mean()
    obs = Z.mean(axis=1, keepdims=True) + target * (spread + VSIZE_PRESETS[source]) / np.sqrt(n_states)
    return lambda _state: (Z, obs)


NO_ENSEMBLE = lambda _state: (None, None)

# sanity: the helper lands where it claims
Z, obs = ensemble_with_innovation(3.0)(None)
print(f"requested 3.0 -> measured "
      f"{checks.normalized_innovation(Z, obs, VSIZE_PRESETS['mapped_perimeter']):.2f}")

requested 3.0 -> measured 3.00


The agent needs `Signals`, which the observation agent normally supplies. This helper builds
one directly so each action can be triggered in isolation.

In [21]:
def signals(**kw) -> Signals:
    base = dict(normalized_innovation=0.8, agrees_with_firms=None, contains_previous=True,
                growth_plausible=True, obs_age_min=5.0, source="mapped_perimeter",
                wind_shift=False, consecutive_disagreements=0)
    base.update(kw)
    return Signals(**base)


def assimilate(policy, sig, innovation=0.8, **over):
    st = base_state(signals=sig, **over)
    node = make_assimilation_agent(policy, load_ensemble=ensemble_with_innovation(innovation))
    return node(st)

### 3.1 `FixedPolicy` — the baseline update

In [22]:
out = assimilate(AsFixed(), signals())
show(out["assim_verdict"], "")
print(f"\nsteps.csv from outputs/Border_2_Real: vsize=300 wsize=100 on every step")

action=assimilate vsize_m=300.0 wsize_m=100.0 inflation=1.0
    FixedPolicy: plain EnKF update with baseline noise (vsize=300, wsize=100).

steps.csv from outputs/Border_2_Real: vsize=300 wsize=100 on every step


### 3.2 `RulePolicy` — all four actions

In [23]:
scenarios = [
    ("quiet cycle",              signals(),                                        0.8,  {}),
    ("surprising, checks clean", signals(),                                        5.0,  {}),
    ("containment failed",       signals(contains_previous=False),                 0.8,  {}),
    ("failed + wind shift",      signals(contains_previous=False, wind_shift=True), 5.0,
     {"weather_verdict": WeatherVerdict(action="wind_shift_detected", wind_shift=True)}),
    ("3-step drift streak",      signals(consecutive_disagreements=3),             5.0,  {}),
]

rows = []
for label, sig, innov, over in scenarios:
    v = assimilate(AsRule(), sig, innov, **over)["assim_verdict"]
    rows.append({"scenario": label, "action": v.action, "vsize_m": v.vsize_m,
                 "inflation": v.inflation, "reason": v.reason})
pd.DataFrame(rows).set_index("scenario")

,action,vsize_m,inflation,reason
scenario,,,,
quiet cycle,assimilate,200.0,1.0,Plain update: innovation 0.80; all checks consistent.
"surprising, checks clean",inflate_and_assimilate,200.0,1.4,Inflate and assimilate: innovation 5.00 (> 2.0) is surprising but ...
containment failed,skip,200.0,1.0,Skip: an independent check rejects this observation (does not cont...
failed + wind shift,inflate_and_assimilate,200.0,1.4,Inflate and assimilate: innovation 5.00 (> 2.0) is surprising but ...
3-step drift streak,reinitialize_from_obs,200.0,1.0,"Reinitialize: the forecast has disagreed 3 steps in a row (>= 3), ..."


Two rows are worth pausing on.

**`containment failed` → `skip`** and **`failed + wind shift` → `inflate_and_assimilate`**
are the same failed check with different context. Without a wind shift the observation is
probably wrong, so it is dropped. With one, the jump is believable, so the prior spread is
widened instead and the observation is used.

**`surprising, checks clean` → `inflate_and_assimilate`** is the other route to inflation: no
wind shift, but every independent check passed, so a large innovation means the *forecast*
missed, not the observation.

### 3.3 Priority order — a drift streak outranks inflation

In [24]:
for streak in (0, 2, 3, 5):
    v = assimilate(AsRule(), signals(consecutive_disagreements=streak), 9.0)["assim_verdict"]
    print(f"  streak={streak}  innovation=9.0  ->  {v.action}")

  streak=0  innovation=9.0  ->  inflate_and_assimilate
  streak=2  innovation=9.0  ->  inflate_and_assimilate
  streak=3  innovation=9.0  ->  reinitialize_from_obs
  streak=5  innovation=9.0  ->  reinitialize_from_obs


Below the threshold a big innovation means inflate; at or above it, the pattern says the
model has drifted and no amount of inflation will fix it, so the state restarts from the
observation.

### 3.4 Guardrail — never skip with nothing to fall back on

In [25]:
out = assimilate(AsRule(), signals(contains_previous=False), 0.8, prev_obs_path=None)
show(out["assim_verdict"], "")
print(f"\nlogged as overridden  {out['log'][0].overridden_by_guardrail}")
print(f"vsize widened to      {out['assim_verdict'].vsize_m:.0f} m "
      f"(synthetic preset = {VSIZE_PRESETS['synthetic_perimeter']:.0f})")

action=assimilate vsize_m=500.0 wsize_m=100.0 inflation=1.0
    Guardrail: skipping was requested but there is no established state to keep, so the observation is assimilated with a widened observation error instead.

logged as overridden  True
vsize widened to      500 m (synthetic preset = 500)


Skipping the only observation of a cycle leaves the state un-updated; on the first cycle it
would never be initialised at all. The node assimilates with a widened observation error
instead — the observation still moves the state, just less.

### 3.5 `vsize_m` follows the source

In [26]:
rows = []
for source in ("mapped_perimeter", "synthetic_perimeter", "firms_viirs", "firms_modis"):
    node = make_assimilation_agent(
        AsRule(), load_ensemble=ensemble_with_innovation(0.8, source))
    v = node(base_state(signals=signals(source=source)))["assim_verdict"]
    rows.append({"source": source, "vsize_m": v.vsize_m, "preset": VSIZE_PRESETS[source]})
pd.DataFrame(rows).set_index("source")

,vsize_m,preset
source,,
mapped_perimeter,200.0,200
synthetic_perimeter,500.0,500
firms_viirs,750.0,750
firms_modis,1000.0,1000


Larger `vsize_m` means the EnKF trusts the observation less and moves the state less far
toward it. A MODIS hotspot (~1 km pixels) should not drag the perimeter as hard as a
hand-mapped one.

### 3.6 A missing innovation is not a low innovation

Easy bug to write: if no ensemble was available, `normalized_innovation` keeps whatever was
in `Signals` — possibly a high value from an earlier cycle — and a stale number drives a real
decision. The node sets `cache["innovation_available"]` so the policy can tell the two apart.

In [27]:
stale_high = signals(normalized_innovation=9.9)   # would trigger inflation if believed

with_ens = make_assimilation_agent(AsRule(), load_ensemble=ensemble_with_innovation(9.9))
no_ens   = make_assimilation_agent(AsRule(), load_ensemble=NO_ENSEMBLE)

for label, node in [("ensemble present", with_ens), ("ensemble missing", no_ens)]:
    out = node(base_state(signals=stale_high))
    print(f"{label:18s} -> {out['assim_verdict'].action:24s} "
          f"available={out['cache']['innovation_available']}")
    print(f"{'':18s}    {out['assim_verdict'].reason}")

ensemble present   -> inflate_and_assimilate   available=True
                      Inflate and assimilate: innovation 9.90 (> 2.0) is surprising but believable because every independent check passed; widening the prior spread by 1.4x.
ensemble missing   -> assimilate               available=False
                      Plain update: innovation unavailable (no ensemble); all checks consistent.


### 3.7 `drift_history` feeds the streak counter

`compute_signals` turns this history into `consecutive_disagreements` by counting the tail
above 1.5. Closing that loop by hand below — recomputing the streak from the history each
cycle, the way the observation agent does — shows rule 2 taking over from rule 3 once the
pattern is unmistakable.

In [28]:
def streak_from(history: list[float], threshold: float = 1.5) -> int:
    # Length of the run of disagreements at the end of the history.
    n = 0
    for v in reversed(history):
        if v <= threshold:
            break
        n += 1
    return n


history: list[float] = []
rows = []
for cycle, innov in enumerate([0.7, 1.2, 4.0, 5.5, 6.0, 3.0, 0.9, 0.8], start=1):
    streak = streak_from(history)
    st = base_state(signals=signals(consecutive_disagreements=streak),
                    drift_history=history)
    out = make_assimilation_agent(
        AsRule(), load_ensemble=ensemble_with_innovation(innov, seed=cycle))(st)
    history = out["drift_history"]
    rows.append({"cycle": cycle, "innovation": innov, "streak_on_entry": streak,
                 "action": out["assim_verdict"].action,
                 "history": [round(x, 1) for x in history]})

pd.DataFrame(rows).set_index("cycle")

,innovation,streak_on_entry,action,history
cycle,,,,
1,0.7,0,assimilate,[0.7]
2,1.2,0,assimilate,"[0.7, 1.2]"
3,4.0,0,inflate_and_assimilate,"[0.7, 1.2, 4.0]"
4,5.5,1,inflate_and_assimilate,"[0.7, 1.2, 4.0, 5.5]"
5,6.0,2,inflate_and_assimilate,"[0.7, 1.2, 4.0, 5.5, 6.0]"
6,3.0,3,reinitialize_from_obs,"[0.7, 1.2, 4.0, 5.5, 6.0, 3.0]"
7,0.9,4,reinitialize_from_obs,"[1.2, 4.0, 5.5, 6.0, 3.0, 0.9]"
8,0.8,0,assimilate,"[4.0, 5.5, 6.0, 3.0, 0.9, 0.8]"


Read `streak_on_entry` against `action`. Cycles 3–5 inflate while the streak is still
building. By cycle 6 it has reached 3, and the agent stops trying to absorb the observations
and restarts the state from the latest one instead — the model, not the data, is the problem.

Cycle 7's observation agrees (innovation 0.9), but the streak is read *on entry*, so it is
still 4 and the agent reinitialises once more. Only on cycle 8, with the agreement now in the
history, does the streak fall to 0 and normal updates resume. That one-cycle lag is inherent
to deciding from history rather than from the current observation alone.

The history is bounded, so a streak can never be inferred from arbitrarily old evidence:

In [29]:
print(f"DRIFT_HISTORY_LEN = {DRIFT_HISTORY_LEN}")
print(f"kept after 8 cycles: {len(history)} entries -> {[round(x, 1) for x in history]}")

DRIFT_HISTORY_LEN = 6
kept after 8 cycles: 6 entries -> [4.0, 5.5, 6.0, 3.0, 0.9, 0.8]


---

## 4. How `Signals` fills up

`Signals` is not computed in one place. The pipeline order makes that impossible: the
observation agent runs before the forecast exists, so it cannot know the innovation. So the
object is filled in three stages, each agent adding what it can see.

| Stage | Node | Adds |
|---|---|---|
| 1 | `observation` | every forecast-independent check, `obs_age_min`, `source`, and a **lagged** `wind_shift` |
| 2 | `weather` | overwrites `wind_shift` for the window about to be forecast |
| 3 | `assimilation` | `normalized_innovation`, once the ensemble exists |

Running the three nodes by hand on one state shows the handoff:

In [30]:
st = base_state(event(1, P_GROWN),
                cache={"good_obs": [event(0, P_PREV).model_dump(mode="json")],
                       "weather_prev": CALM, "weather_last": CALM})

stages = []

# stage 1 — observation
out = make_observation_agent(ObsRule())(st)
st = {**st, **out}
stages.append(("1. after observation", st["signals"]))

# stage 2 — weather turns sharply, so the lagged value is replaced
out = make_weather_agent(WxRule(), fetch=lambda s, e: SHIFT)(st)
st = {**st, **out}
stages.append(("2. after weather", st["signals"]))

# stage 3 — the ensemble arrives
out = make_assimilation_agent(AsRule(), load_ensemble=ensemble_with_innovation(4.2))(st)
st = {**st, **out}
stages.append(("3. after assimilation", st["signals"]))

pd.DataFrame({label: signal_row(sig) for label, sig in stages})

,1. after observation,2. after weather,3. after assimilation
innovation,0.0,0.0,4.2
contains_prev,True,True,True
growth_ok,True,True,True
firms,None,None,None
age_min,5,5,5
wind_shift,False,True,True
streak,0,0,0
FAILED,False,False,False


In [31]:
print("weather verdict :", st["weather_verdict"].action)
print("assim verdict   :", st["assim_verdict"].action)
print("                 ", st["assim_verdict"].reason)

weather verdict : wind_shift_detected
assim verdict   : inflate_and_assimilate
                  Inflate and assimilate: innovation 4.20 (> 2.0) is surprising but believable because a wind shift was detected; widening the prior spread by 1.4x.


The chain is the point. The observation agent saw calm-to-calm and reported no shift. The
weather agent fetched the new window, found a reversal, and overwrote the field. The
assimilation agent then read *that* value, found a surprising innovation, and inflated rather
than skipping — a decision no single agent had enough information to make alone.

---

## 5. Replay: baseline vs agentic

The comparison the experiment rests on. Same observations, same weather, same innovations —
only the policies change.

- **Fixed** — accept everything, baseline FARSITE settings, plain EnKF update every step.
  Reproduces `outputs/Border_2_Real`.
- **Rule** — the policies from sections 1–3.

`forecast` is still `placeholder_forecast` (the real one needs `adapter.load_state`, see
`steps.forecast`), so the innovations are scripted rather than measured. Everything else is
the real pipeline: real graph, real routing, real agents.

In [32]:
# Four observations, 3 h apart. The wind turns before observation 3, and the fire
# jumps in response — the innovation spikes on that cycle.
EVENTS     = [event(i, P_GROWN) for i in range(4)]
WEATHER    = [CALM, CALM, SHIFT, SHIFT]
INNOVATION = [0.6, 0.9, 4.5, 1.1]


def replay(obs_policy, assim_policy):
    queue = list(WEATHER)

    def fetch(start, end):
        return queue.pop(0) if queue else WEATHER[-1]

    def load(state):
        target = INNOVATION[min(state["step"] - 1, len(INNOVATION) - 1)]
        return ensemble_with_innovation(target, seed=state["step"])(state)

    nodes = placeholder_nodes()
    nodes["observation"]  = make_observation_agent(obs_policy)
    nodes["weather"]      = make_weather_agent(WxRule(), fetch=fetch)
    nodes["assimilation"] = make_assimilation_agent(assim_policy, load_ensemble=load)

    return run_pipeline(
        build_pipeline(nodes),
        initial_state(EVENTS, OperatorRequest(deadline_min=60), str(TMP / "replay.npz")),
    )


baseline = replay(ObsFixed(), AsFixed())
agentic  = replay(ObsRule(),  AsRule())
print(f"baseline: {len(baseline['metrics'])} cycles")
print(f"agentic : {len(agentic['metrics'])} cycles")

baseline: 4 cycles
agentic : 4 cycles


In [33]:
comparison = pd.DataFrame({
    "cycle":          [m["step"] for m in agentic["metrics"]],
    "mean wind (mph)": [np.mean(w["wind_speeds"]) for w in WEATHER],
    "innovation":     INNOVATION,
    "baseline assim": [m["assim_action"] for m in baseline["metrics"]],
    "agentic assim":  [m["assim_action"] for m in agentic["metrics"]],
    "weather":        [m["weather_action"] for m in agentic["metrics"]],
}).set_index("cycle")
comparison

,mean wind (mph),innovation,baseline assim,agentic assim,weather
cycle,,,,,
1,8.5,0.6,assimilate,assimilate,normal
2,8.5,0.9,assimilate,assimilate,normal
3,31.0,4.5,assimilate,inflate_and_assimilate,wind_shift_detected
4,31.0,1.1,assimilate,assimilate,normal


Cycle 3 is the whole story. The wind reverses, the observation lands far from the forecast,
and the two configurations diverge: the baseline applies the same plain update it applies
everywhere, while the agentic run widens the prior spread first so the update can move
further without the filter dismissing the observation as an outlier.

By cycle 4 conditions have settled and both do a plain update again — the agentic policies
only intervene where the situation calls for it.

In [34]:
log = pd.DataFrame([{
    "cycle": i // 3 + 1,
    "node": r.node,
    "policy": r.policy,
    "action": r.decision.get("action"),
    "guardrail": r.overridden_by_guardrail,
    "latency_s": r.agent_latency_s,
} for i, r in enumerate(agentic["log"])])
log.set_index(["cycle", "node"])

policy                  action  guardrail  latency_s
cycle node                                                                  
1     observation   RulePolicy                  accept      False     0.0023
      weather       RulePolicy                  normal      False     0.0001
      assimilation  RulePolicy              assimilate      False     0.0002
2     observation   RulePolicy                  accept      False     0.0036
      weather       RulePolicy                  normal      False     0.0001
      assimilation  RulePolicy              assimilate      False     0.0002
3     observation   RulePolicy                  accept      False     0.0036
      weather       RulePolicy     wind_shift_detected      False     0.0001
      assimilation  RulePolicy  inflate_and_assimilate      False     0.0002
4     observation   RulePolicy                  accept      False     0.0035
      weather       RulePolicy                  normal      False     0.0001
      assimilation  RulePolicy              assimilate      False     0.0002

Every decision in the run, with its policy and latency. This is the raw material for
`metrics.py` (§10): decisions per agent, guardrail override rate, and agent latency — which
counts against the operator's deadline once LLM policies replace the rule-based ones.

In [35]:
print("decisions per node")
print(log.groupby("node")["action"].value_counts().to_string())
print(f"\nguardrail overrides : {int(log['guardrail'].sum())} of {len(log)} decisions")
print(f"total agent latency : {log['latency_s'].sum() * 1000:.1f} ms "
      f"(rule-based; LLM policies will dominate this)")
print(f"\nfinal weather cache : {sorted(k for k in agentic['cache'] if k.startswith('weather'))}")
print(f"final drift_history : {[round(x, 2) for x in agentic['drift_history']]}")

decisions per node
node          action                
assimilation  assimilate                3
              inflate_and_assimilate    1
observation   accept                    4
weather       normal                    3
              wind_shift_detected       1

guardrail overrides : 0 of 12 decisions
total agent latency : 14.2 ms (rule-based; LLM policies will dominate this)

final weather cache : ['weather_last', 'weather_prev']
final drift_history : [0.6, 0.9, 4.5, 1.1]


---

## What this notebook does not cover

| Gap | Why | Where it lands |
|---|---|---|
| Real FARSITE forecasts | `steps.forecast` raises `NotImplementedError`; `adapter.load_state` / `save_state` now exist, so it is implementable | §8 in `TODO.md` |
| Measured innovations | Scripted above, since they come from the ensemble | follows the forecast step |
| FIRMS agreement | `agrees_with_firms` returns `None` without `cache["hotspot_path"]`, so `firms` is `None` throughout | §3 in `TODO.md` |
| LLM policies | Each is a fourth subclass of the existing `*Policy` ABCs — the nodes do not change | "Then" in `TODO.md` |
| Accuracy metrics | Area error and IoU of the analysis against the observed perimeter | §10 in `TODO.md` |

The assertions behind everything above live in `test_agents.py`; run
`python -m pytest orchestrator/ -q` for the full suite.

In [36]:
import shutil
shutil.rmtree(TMP, ignore_errors=True)
print(f"cleaned up {TMP}")

cleaned up /tmp/agent_demo_6w2mud9c
